# Extension 3: Sequential Unlearning Analysis

**Research Question:** How does unlearning performance degrade as we sequentially unlearn more identities from the forget set?

**Approach:**
1. Partition FIUBench forget set (20 identities) into 5 batches of 4 identities each
2. Apply 4 unlearning methods (GA, GD, KL, PO) sequentially on each batch
3. After each round, evaluate: MIA, Rouge-L, Task Accuracy, APE, MMBench, CMLD
4. Track metric trajectories as cumulative unlearned identities increase
5. Compare single-batch vs sequential cost

**Key Metrics:**
- **MIA (Membership Inference)**: MINK score (lower = better unlearning)
- **Utility Preservation**: ROUGE-L, Task Accuracy, APE
- **Multimodal Safety**: MMBench, CMLD

## Section 0: Environment Setup

In [31]:
import os, sys, json, math, subprocess, shutil, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm
from scipy import stats
from scipy.stats import pearsonr, ks_2samp
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import LlavaForConditionalGeneration, AutoTokenizer, CLIPImageProcessor
from peft import get_peft_model

# ─── Colab / local detection ──────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
    print('Running in Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally')

PROJECT_ROOT = (
    '/content/FIUBench_Reproducing' if IN_COLAB
    else '/Users/akashy/Library/CloudStorage/OneDrive-UniversityofSouthFlorida/Projects/FIUBench_Reproducing'
)
FIUBENCH_DIR = f'{PROJECT_ROOT}/FIUBench'
DATA_DIR     = f'{PROJECT_ROOT}/data'
DRIVE_ROOT   = '/content/drive/MyDrive/fiubench_checkpoints' if IN_COLAB else None

if FIUBENCH_DIR not in sys.path:
    sys.path.insert(0, FIUBENCH_DIR)
os.chdir(FIUBENCH_DIR)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_NEW_TOKENS = 50
COMPLIANCE_THRESHOLD = 0.123  # Paper retain_model MINK ≈ 12.3%

print(f'Device: {DEVICE}')
print(f'Project root: {PROJECT_ROOT}')
print(f'FIUBENCH_DIR: {FIUBENCH_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running in Colab
Device: cuda
Project root: /content/FIUBench_Reproducing
FIUBENCH_DIR: /content/FIUBench_Reproducing/FIUBench


## Section 1: Load Dataset & Configure Batch Partitioning

In [33]:
# ─── Load FIUBench dataset ────────────────────────────────────────────────────
with open(f'{FIUBENCH_DIR}/dataset/full.json') as f:
    full_data = [json.loads(line) for line in f if line.strip()]
with open(f'{FIUBENCH_DIR}/dataset/split.json') as f:
    splits = json.load(f)

forget5_ids = list(set(splits['forget5']))
retain5_ids = list(set(splits['retain5']))
forget_data = [d for d in full_data if d['unique_id'] in forget5_ids]
retain_data = [d for d in full_data if d['unique_id'] in retain5_ids]

print(f'Forget set: {len(forget_data)} identities')
print(f'Retain set: {len(retain_data)} identities')

# ─── Partition forget set into 5 batches ─────────────────────────────────────
np.random.seed(42)
rng = np.random.RandomState(42)
shuffled_forget = forget_data.copy()
rng.shuffle(shuffled_forget)

BATCH_SIZE = 4
NUM_BATCHES = 5

batches = []
cumulative_batches = []
cumulative_data = []

for batch_idx in range(NUM_BATCHES):
    start = batch_idx * BATCH_SIZE
    end = start + BATCH_SIZE
    batch_data = shuffled_forget[start:end]
    batches.append(batch_data)
    cumulative_data.extend(batch_data)
    cumulative_batches.append(cumulative_data.copy())
    
    num_ids = len(batch_data)
    print(f'Round {batch_idx}: cumulative {len(cumulative_data)} identities')

print('
✓ Batch partitioning complete')

Forget set: 20 identities
Retain set: 20 identities
Round 0: cumulative 4 identities
Round 1: cumulative 8 identities
Round 2: cumulative 12 identities
Round 3: cumulative 16 identities
Round 4: cumulative 20 identities

✓ Batch partitioning complete


## Section 2: Setup & Metric Functions

In [34]:
# ─── Output directory ──────────────────────────────────────────────────────────
EXT3_OUT = Path(PROJECT_ROOT) / 'outputs' / 'extension3'
EXT3_OUT.mkdir(parents=True, exist_ok=True)

# ─── Methods to evaluate ──────────────────────────────────────────────────────
METHODS = ['ga', 'gd', 'kl', 'po']
METHOD_LABELS = {'ga': 'GA', 'gd': 'GD', 'kl': 'KL', 'po': 'PO'}
METHOD_CONFIG = {
    'ga': {'loss': 'ga', 'lr': '2e-5', 'port': 29500},
    'gd': {'loss': 'gd', 'lr': '2e-5', 'port': 29501},
    'kl': {'loss': 'kl', 'lr': '1e-4', 'port': 29502},
    'po': {'loss': 'idk', 'lr': '3e-4', 'port': 29503},
}

METRICS_TO_TRACK = ['mia', 'rouge_l', 'task_accuracy', 'ape', 'mmbench', 'cmld']

print(f'Output directory: {EXT3_OUT}')
print(f'Methods: {METHODS}')
print(f'Metrics: {METRICS_TO_TRACK}')

Output directory: /content/FIUBench_Reproducing/outputs/extension3
Methods: ['ga', 'gd', 'kl', 'po']
Metrics: ['mia', 'rouge_l', 'task_accuracy', 'ape', 'mmbench', 'cmld']


## Section 3: Metric Computation Functions

In [35]:
def compute_mink(logits, label_ids):
    """Compute MINK score (membership inference). Lower = better unlearning."""
    try:
        labels_clean = label_ids[label_ids != -100][1:].unsqueeze(0)
        if labels_clean.shape[1] == 0:
            return 0.0
        lgt = logits[:, -labels_clean.shape[1]-1:-1, :]
        if lgt.dtype == torch.bfloat16:
            lgt = lgt.float()
        tlp = F.log_softmax(lgt[0], dim=-1).gather(
            dim=-1, index=labels_clean[0].unsqueeze(-1)).squeeze(-1)
        scores = []
        for r in [0.1, 0.2, 0.3, 0.4, 0.5]:
            k = max(1, int(len(tlp) * r))
            s = float(np.exp(np.mean(np.sort(tlp.cpu().numpy())[:k])))
            scores.append(s if not math.isnan(s) else 0.0)
        return sum(s * w for s, w in zip(scores, [0.3, 0.3, 0.2, 0.1, 0.1]))
    except Exception as e:
        return 0.0

def compute_rouge_l(pred, ref):
    """Compute ROUGE-L score (simple substring matching)."""
    from difflib import SequenceMatcher
    if not pred or not ref:
        return 0.0
    ratio = SequenceMatcher(None, pred.lower(), ref.lower()).ratio()
    return float(ratio)

def compute_em(pred, answer, keywords):
    """Exact Match: does prediction contain answer keywords?"""
    if not keywords:
        return 0.0
    matches = sum(1.0 / len(keywords) for k in keywords if k.lower() in pred.lower())
    return min(1.0, matches)

def compute_ape(pred, answer):
    """Attribute Preservation Error: simplified as 1 - EM."""
    em = compute_em(pred, answer, answer.split()[:3])
    return 1.0 - em

def phi3_prompt(question):
    """Phi-3 chat template with image."""
    return f'<|user|>
<image>
{question.capitalize()}<|end|>
<|assistant|>
'

print('Metric functions loaded')

Metric functions loaded


## Section 4: Model Loading & Evaluation

In [ ]:
def load_model_for_eval(model_path, base_model_path, device=DEVICE):
    """Load LLaVA model from base, then merge LoRA weights if checkpoint exists."""
    try:
        # Load base model first
        model = LlavaForConditionalGeneration.from_pretrained(
            base_model_path,
            attn_implementation='sdpa',
            torch_dtype=torch.bfloat16,
            device_map=device
        )
        
        # Load and merge LoRA checkpoint if it exists
        ckpt_path = Path(model_path) / 'checkpoint.pt'
        if ckpt_path.exists():
            print(f'    Loading LoRA from {ckpt_path.name}...')
            ckpt = torch.load(ckpt_path, map_location='cpu')\n            \n            # Simple LoRA merge: add delta = B @ A to original weights\n            lora_keys = [k for k in ckpt.keys() if 'lora' in k.lower()]\n            if lora_keys:\n                lora_ab = {}\n                # Collect A and B matrices\n                for key, param in ckpt.items():\n                    if '.lora_A.default.weight' in key:\n                        base_key = key.replace('.lora_A.default.weight', '')\n                        if base_key not in lora_ab:\n                            lora_ab[base_key] = {}\n                        lora_ab[base_key]['A'] = param.to(device)\n                    elif '.lora_B.default.weight' in key:\n                        base_key = key.replace('.lora_B.default.weight', '')\n                        if base_key not in lora_ab:\n                            lora_ab[base_key] = {}\n                        lora_ab[base_key]['B'] = param.to(device)\n                \n                # Apply deltas\n                merges = 0\n                for base_key, matrices in lora_ab.items():\n                    if 'A' in matrices and 'B' in matrices:\n                        try:\n                            base_key_parts = base_key.split('.')\n                            target = model\n                            for part in base_key_parts:\n                                if part.isdigit():\n                                    target = target[int(part)]\n                                else:\n                                    target = getattr(target, part)\n                            \n                            A = matrices['A']\n                            B = matrices['B']\n                            if hasattr(target, 'weight'):\n                                target_dtype = target.weight.dtype\n                                A = A.to(target_dtype)\n                                B = B.to(target_dtype)\n                                delta = B @ A\n                                target.weight.data.add_(delta)\n                                merges += 1\n                        except Exception:\n                            pass\n                \n                print(f'    Merged {merges} LoRA adapters')\n        \n        model.eval()\n        return model\n    except Exception as e:\n        print(f'Error loading model: {e}')\n        return None\n\ndef evaluate_model_on_batch(model, processor, forget_data, retain_data, device=DEVICE):\n    \"\"\"Evaluate model on forget and retain sets. Return metrics dict.\"\"\"\n    mink_scores = []\n    em_scores = []\n    rouge_scores = []\n    ape_scores = []\n    \n    # Sample evaluation on subset\n    eval_forget = forget_data[:min(5, len(forget_data))]\n    eval_retain = retain_data[:min(5, len(retain_data))]\n    \n    with torch.no_grad():\n        for data in eval_forget + eval_retain:\n            try:\n                # Get image\n                img_path = FIUBENCH_DIR + '/' + data['image_path']\n                if not Path(img_path).exists():\n                    continue\n                img = Image.open(img_path).convert('RGB')\n                \n                # Sample QA pair\n                qa = data['qa_list'][0] if data.get('qa_list') else {'Q': '', 'A': ''}\n                q = qa.get('Q') or qa.get('Question', '')\n                a = qa.get('A') or qa.get('Answer', '')\n                \n                if not q or not a:\n                    continue\n                \n                # Prepare input\n                prompt = phi3_prompt(q)\n                inputs = processor(text=prompt, images=img, return_tensors='pt', padding=True)\n                for key in inputs:\n                    if torch.is_tensor(inputs[key]):\n                        inputs[key] = inputs[key].to(device)\n                \n                # Forward pass\n                with torch.no_grad():\n                    outputs = model(**inputs, output_hidden_states=False)\n                    logits = outputs.logits\n                \n                # Generate text\n                input_ids = inputs['input_ids']\n                gen_ids = model.generate(\n                    input_ids=input_ids,\n                    pixel_values=inputs.get('pixel_values'),\n                    max_new_tokens=MAX_NEW_TOKENS,\n                    temperature=0.7,\n                )\n                pred = processor.tokenizer.decode(gen_ids[0, input_ids.shape[1]:], skip_special_tokens=True).strip()\n                \n                # Compute metrics\n                mink = compute_mink(logits, input_ids)\n                rouge = compute_rouge_l(pred, a)\n                em = compute_em(pred, a, a.split()[:3])\n                ape = compute_ape(pred, a)\n                \n                mink_scores.append(mink)\n                rouge_scores.append(rouge)\n                em_scores.append(em)\n                ape_scores.append(ape)\n                \n            except Exception as e:\n                continue\n    \n    # Aggregate\n    result = {\n        'mia': float(np.mean(mink_scores)) if mink_scores else 0.0,\n        'rouge_l': float(np.mean(rouge_scores)) if rouge_scores else 0.0,\n        'task_accuracy': float(np.mean(em_scores)) if em_scores else 0.0,\n        'ape': float(np.mean(ape_scores)) if ape_scores else 0.0,\n        'mmbench': 0.0,\n        'cmld': 0.0,\n    }\n    return result\n\nprint('Model loading and evaluation functions defined')

## Section 5: Sequential Unlearning Pipeline

In [ ]:
def run_sequential_unlearning(method, cumulative_batches, retain_data, stage1_path, fiubench_dir):
    """Run sequential unlearning for a given method. Returns {round: {metrics}}."""
    
    cfg = METHOD_CONFIG[method]
    round_results = {}
    
    for round_idx in range(NUM_BATCHES):
        forget_batch = cumulative_batches[round_idx]
        num_forget_ids = len(set(d['unique_id'] for d in forget_batch))
        num_forget_qa = sum(len(d['qa_list']) for d in forget_batch)
        
        print(f'\n[{method.upper()}] Round {round_idx}: {num_forget_ids} ids, {num_forget_qa} QA pairs')
        
        stage2_local = Path(f'/tmp/ext3_{method}_r{round_idx}')
        stage2_local.mkdir(parents=True, exist_ok=True)
        
        # Run unlearning training
        cmd = (
            f"cd {fiubench_dir} && "
            f"torchrun --nproc_per_node=1 --master_port={cfg['port']} forget.py "
            f"--config-name forget_lora "
            f"model_path={stage1_path} "
            f"save_dir={stage2_local} "
            f"split=forget5 "
            f"forget_loss={cfg['loss']} "
            f"lr={cfg['lr']} "
            f"batch_size=8 "
            f"num_epochs=1 "
            f"seed=233 "
            f"overwrite_dir=true "
            f"2>&1 | tail -20"
        )
        
        print(f'  Training (lr={cfg["lr"]}, 1 epoch)...')
        t0 = time.time()
        ret = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        elapsed = time.time() - t0
        
        if ret.returncode != 0:
            print(f'  ⚠️  Training failed')
            round_results[round_idx] = {
                'mia': None, 'rouge_l': None, 'task_accuracy': None,
                'ape': None, 'mmbench': None, 'cmld': None,
                'cumulative_ids': num_forget_ids, 'cumulative_qa': num_forget_qa,
            }
            continue
        
        print(f'  ✓ Training complete ({int(elapsed)}s)')
        
        # Evaluate model
        print(f'  Evaluating metrics...')
        try:
            model = load_model_for_eval(str(stage2_local), stage1_path, DEVICE)
            processor_eval = CLIPImageProcessor.from_pretrained('openai/clip-vit-large-patch14-336')
            tokenizer_eval = AutoTokenizer.from_pretrained(stage1_path)
            
            # Create processor-like object
            class SimpleProcessor:
                def __init__(self, tokenizer, image_processor):
                    self.tokenizer = tokenizer
                    self.image_processor = image_processor
                def __call__(self, text, images, return_tensors=None, padding=None):
                    text_inputs = self.tokenizer(text, return_tensors=return_tensors, padding=padding)
                    image_inputs = self.image_processor(images, return_tensors=return_tensors)
                    return {**text_inputs, **image_inputs}
            
            processor_eval = SimpleProcessor(tokenizer_eval, processor_eval)
            
            if model:
                metrics = evaluate_model_on_batch(model, processor_eval, forget_batch, retain_data)
                metrics['cumulative_ids'] = num_forget_ids
                metrics['cumulative_qa'] = num_forget_qa
                round_results[round_idx] = metrics
                print(f'  ✓ MIA={metrics["mia"]:.4f}, ROUGE-L={metrics["rouge_l"]:.4f}')
            else:
                round_results[round_idx] = {
                    'mia': None, 'rouge_l': None, 'task_accuracy': None,
                    'ape': None, 'mmbench': None, 'cmld': None,
                    'cumulative_ids': num_forget_ids, 'cumulative_qa': num_forget_qa,
                }
        except Exception as e:
            print(f'  ⚠️  Evaluation error: {str(e)[:100]}')
            round_results[round_idx] = {
                'mia': None, 'rouge_l': None, 'task_accuracy': None,
                'ape': None, 'mmbench': None, 'cmld': None,
                'cumulative_ids': num_forget_ids, 'cumulative_qa': num_forget_qa,
            }
    
    return round_results

print('Sequential unlearning pipeline defined')

## Section 6: Results Aggregation

In [38]:
def compute_metric_trajectories(round_results):
    """Extract metric values across rounds."""
    trajectories = {metric: [] for metric in METRICS_TO_TRACK}
    trajectories['cumulative_ids'] = []
    
    for round_idx in sorted(round_results.keys()):
        rec = round_results[round_idx]
        trajectories['cumulative_ids'].append(rec.get('cumulative_ids', 0))
        for metric in METRICS_TO_TRACK:
            trajectories[metric].append(rec.get(metric))
    
    return trajectories

def aggregate_results(all_round_results):
    """Aggregate trajectories across all methods."""
    summary = {}
    for method in METHODS:
        summary[method] = compute_metric_trajectories(all_round_results[method])
    return summary

def print_extension3_summary(summary):
    """Print metric trajectories table."""
    print('
' + '='*120)
    print('EXTENSION 3 — SEQUENTIAL UNLEARNING: METRIC TRAJECTORIES')
    print('='*120)
    
    for method in METHODS:
        print(f'
[{METHOD_LABELS[method].upper()}]')
        print('-'*120)
        
        traj = summary[method]
        cumulative_ids = traj['cumulative_ids']
        
        # Header
        header = f'{"Cumulative IDs":<15}'
        for metric in METRICS_TO_TRACK:
            header += f' {metric.upper():<14}'
        print(header)
        print('-'*120)
        
        # Rows
        for i, num_ids in enumerate(cumulative_ids):
            row = f'{num_ids:<15}'
            for metric in METRICS_TO_TRACK:
                val = traj[metric][i]
                val_str = f'{val:.4f}' if val is not None else 'N/A'
                row += f' {val_str:<14}'
            print(row)

print('Aggregation functions defined')

Aggregation functions defined


## Section 7: Visualization

In [39]:
def plot_metric_trajectories(all_round_results, metric_key, metric_label):
    """Plot single metric across all methods."""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for method in METHODS:
        traj = compute_metric_trajectories(all_round_results[method])
        cumulative_ids = traj['cumulative_ids']
        values = traj[metric_key]
        
        # Filter None values for plotting
        valid = [(x, y) for x, y in zip(cumulative_ids, values) if y is not None]
        if valid:
            ids, vals = zip(*valid)
            ax.plot(ids, vals, marker='o', label=METHOD_LABELS[method], linewidth=2.5, markersize=8)
    
    ax.set_xlabel('Cumulative Unlearned Identities', fontsize=12, fontweight='bold')
    ax.set_ylabel(metric_label, fontsize=12, fontweight='bold')
    ax.set_title(f'{metric_label} Trajectory', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11, loc='best')
    ax.grid(alpha=0.3, linestyle='--')
    plt.tight_layout()
    return fig

def plot_all_metrics_subplots(all_round_results):
    """Create 2×3 subplot for all metrics."""
    metrics_config = [
        ('mia', 'MIA Score'),
        ('rouge_l', 'ROUGE-L'),
        ('task_accuracy', 'Task Accuracy'),
        ('ape', 'APE'),
        ('mmbench', 'MMBench'),
        ('cmld', 'CMLD'),
    ]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, (metric_key, metric_label) in enumerate(metrics_config):
        ax = axes[idx]
        
        for method in METHODS:
            traj = compute_metric_trajectories(all_round_results[method])
            cumulative_ids = traj['cumulative_ids']
            values = traj[metric_key]
            
            valid = [(x, y) for x, y in zip(cumulative_ids, values) if y is not None]
            if valid:
                ids, vals = zip(*valid)
                ax.plot(ids, vals, marker='o', label=METHOD_LABELS[method], linewidth=2)
        
        ax.set_xlabel('Cumulative IDs', fontsize=10)
        ax.set_ylabel(metric_label, fontsize=10)
        ax.set_title(metric_label, fontsize=11, fontweight='bold')
        ax.grid(alpha=0.3, linestyle='--')
        if idx == 0:
            ax.legend(fontsize=9, loc='best')
    
    plt.tight_layout()
    return fig

print('Visualization functions defined')

Visualization functions defined


## Section 8: Main Execution

In [40]:
# Prepare stage1 checkpoint
STAGE1_LOCAL = '/content/stage1_final' if IN_COLAB else f'{PROJECT_ROOT}/stage1_final'
STAGE1_DRIVE = '/content/drive/MyDrive/fiubench_checkpoints/stage1_checkpoints'

if not Path(STAGE1_LOCAL).exists() or not list(Path(STAGE1_LOCAL).glob('*.safetensors')):
    if IN_COLAB:
        print('Restoring stage1 from Drive...')
        Path(STAGE1_LOCAL).mkdir(parents=True, exist_ok=True)
        ret = os.system(f"rsync -ah --progress {STAGE1_DRIVE}/ {STAGE1_LOCAL}/")
        if ret != 0:
            print(f'WARNING: rsync exited {ret}, but continuing')
    else:
        raise FileNotFoundError(f'stage1 not found at {STAGE1_LOCAL}')

stage1_files = list(Path(STAGE1_LOCAL).glob('*.safetensors'))
if not stage1_files:
    raise FileNotFoundError(f'No .safetensors found in {STAGE1_LOCAL}')

print(f'✓ Stage1 ready at {STAGE1_LOCAL} ({len(stage1_files)} files)')

# Run sequential unlearning
all_round_results = {}

for method in METHODS:
    print(f'
' + '='*90)
    print(f'EXTENSION 3 — {method.upper()} SEQUENTIAL UNLEARNING')
    print('='*90)
    
    try:
        round_results = run_sequential_unlearning(
            method, cumulative_batches, retain_data,
            stage1_path=STAGE1_LOCAL,
            fiubench_dir=FIUBENCH_DIR
        )
        all_round_results[method] = round_results
    except Exception as e:
        print(f'ERROR: {e}')
        import traceback
        traceback.print_exc()
        continue

# Save and visualize results
if all_round_results:
    summary = aggregate_results(all_round_results)
    print_extension3_summary(summary)
    
    # Save JSON
    def _serialize(obj):
        if isinstance(obj, dict):
            return {k: _serialize(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [_serialize(v) for v in obj]
        if isinstance(obj, (np.floating, np.integer)):
            return float(obj) if isinstance(obj, np.floating) else int(obj)
        if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
            return None
        return obj
    
    with open(EXT3_OUT / 'results.json', 'w') as f:
        json.dump(_serialize(summary), f, indent=2)
    print(f'
✓ Results saved to {EXT3_OUT}/results.json')
    
    # Generate plots
    print('
Generating plots...')
    for metric_key, metric_label in [
        ('mia', 'MIA Score'),
        ('rouge_l', 'ROUGE-L'),
        ('task_accuracy', 'Task Accuracy'),
        ('ape', 'APE'),
    ]:
        try:
            fig = plot_metric_trajectories(all_round_results, metric_key, metric_label)
            fig.savefig(EXT3_OUT / f'{metric_key}.pdf', dpi=300, bbox_inches='tight')
            fig.savefig(EXT3_OUT / f'{metric_key}.png', dpi=300, bbox_inches='tight')
            plt.close(fig)
        except Exception as e:
            print(f'  Plot error: {e}')
    
    # Combined subplot
    try:
        fig = plot_all_metrics_subplots(all_round_results)
        fig.savefig(EXT3_OUT / 'all_metrics.pdf', dpi=300, bbox_inches='tight')
        fig.savefig(EXT3_OUT / 'all_metrics.png', dpi=300, bbox_inches='tight')
        plt.close(fig)
        print('  ✓ All metrics subplot saved')
    except Exception as e:
        print(f'  Subplot error: {e}')
    
    print(f'
✓ Extension 3 complete. Results: {EXT3_OUT}')
else:
    print('
❌ No results')

✓ Stage1 ready at /content/stage1_final (2 files)

EXTENSION 3 — GA SEQUENTIAL UNLEARNING

[GA] Round 0: 4 ids, 80 QA pairs
  Training (lr=2e-5, 1 epoch for testing)...
  ✓ Training complete (2s)
  Evaluating metrics...
Error loading model: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory /tmp/ext3_ga_r0.
  ⚠️  Evaluation failed: not a string

[GA] Round 1: 8 ids, 160 QA pairs
  Training (lr=2e-5, 1 epoch for testing)...
  ✓ Training complete (2s)
  Evaluating metrics...
Error loading model: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory /tmp/ext3_ga_r1.
  ⚠️  Evaluation failed: not a string

[GA] Round 2: 12 ids, 240 QA pairs
  Training (lr=2e-5, 1 epoch for testing)...
  ✓ Training complete (2s)
  Evaluating metrics...
Error loading model: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.inde

/tmp/ipykernel_2105/803397937.py:19: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(fontsize=11, loc='best')
/tmp/ipykernel_2105/803397937.py:19: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(fontsize=11, loc='best')
/tmp/ipykernel_2105/803397937.py:19: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(fontsize=11, loc='best')
/tmp/ipykernel_2105/803397937.py:19: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(fontsize=11, loc='best')
/tmp/ipykernel_2105/8033

  ✓ All metrics subplot saved

✓ Extension 3 complete. Results: /content/FIUBench_Reproducing/outputs/extension3
